# Gaussian Beam in Water — CPU Simulation

Simulates a paraxial Gaussian beam propagating 16 m through clear ocean water at 532 nm
using the NumPy (CPU) backend. Demonstrates the full Photonator workflow:
beam → medium → phase function → receiver → result analysis.

**Physical scenario**

| Parameter | Value | Source |
|---|---|---|
| Wavelength | 532 nm | Nd:YAG 2ω |
| Absorption coeff μ_a | 0.0088 m⁻¹ | Pope & Fry (1997) |
| Scattering coeff μ_s | 0.037 m⁻¹ | Petzold clear ocean |
| Attenuation coeff c | 0.0458 m⁻¹ | μ_a + μ_s |
| Asymmetry g | 0.93 | Petzold clear ocean |
| Beam waist w₀ | 1 mm | |
| Half-angle divergence | 0.75 mrad | |
| Receiver range | 16 m | ≈ 0.73 attenuation lengths |
| Aperture diameter | 0.8 m | |
| FOV | π/2 rad (90°) | |


## 1. Imports

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np

from photonator import Simulation
from photonator.beam.gaussian import GaussianBeam
from photonator.core.receiver import Receiver
from photonator.io.hdf5 import load_hdf5, save_hdf5
from photonator.media.water import Water
from photonator.phase_functions.petzold import PetzoldPhaseFunction

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

## 2. Build simulation components

In [ ]:
# ── Medium: clear ocean water at 532 nm ──────────────────────────────────────
water = Water(
    mu_a_per_m=0.0088,          # Pope & Fry (1997) absorption
    mu_s_per_m=0.037,           # Petzold clear ocean scattering
    g=0.93,                     # asymmetry parameter
    turbidity=1.0,              # no additional turbidity
    wavelength_nm=532.0,
)

c = water.mu_a_per_m + water.mu_s_per_m
att_length_m = 1.0 / c
print(f"Attenuation coefficient c = {c:.4f} m⁻¹  →  1/c = {att_length_m:.1f} m")

# ── Beam: paraxial Gaussian with thin-lens divergence ────────────────────────
beam = GaussianBeam(
    w0_m=0.001,                 # 1 mm beam waist
    half_angle_divergence_rad=0.00075,  # 0.75 mrad half-angle
)

# ── Phase function: Petzold measured ocean VSF (clear water) ─────────────────
phase_fn = PetzoldPhaseFunction(water_type="clear")

# ── Receiver at 16 m range ───────────────────────────────────────────────────
RANGE_M = 16.0
receiver = Receiver(
    receiver_z_m=RANGE_M,
    aperture_m=0.8,             # 0.8 m diameter
    fov_rad=np.pi / 2,          # 90° full FOV
)

print(f"Receiver at z = {RANGE_M} m  ({RANGE_M / att_length_m:.2f} attenuation lengths)")

## 3. Run the simulation (CPU backend)

In [ ]:
N_PHOTONS = 100_000   # photons per batch
N_BATCHES = 5         # independent batches (500 k total)

sim = Simulation(
    medium=water,
    beam=beam,
    phase_fn=phase_fn,
    receiver=receiver,
    n_photons=N_PHOTONS,
    n_batches=N_BATCHES,
    backend="cpu",           # NumPy vectorized CPU
    store_photons=True,      # keep raw photon arrays for plotting
    seed=42,
)

print(f"Launching {N_PHOTONS * N_BATCHES:,} photons across {N_BATCHES} batches …")
result = sim.run()
print(f"Done in {result.elapsed_s:.1f} s")

## 4. Results summary

In [ ]:
# Beer-Lambert ballistic prediction (unscattered fraction)
beer_lambert = np.exp(-c * RANGE_M)

# Detected fraction (includes scattered photons within aperture/FOV)
detected_fraction = result.total_power / result.n_photons

print("=" * 50)
print("SIMULATION RESULTS")
print("=" * 50)
print(f"Total photons launched   : {result.n_photons:>12,}")
print(f"Photons detected         : {result.total_packets:>12,}")
print(f"Total received power     : {result.total_power:>12.4f}")
print(f"Detected fraction        : {detected_fraction:>12.6f}")
print(f"Beer-Lambert prediction  : {beer_lambert:>12.6f}  (ballistic only)")
print(f"Scattered excess         : {detected_fraction - beer_lambert:>+12.6f}")
print("-" * 50)
print(f"Mean arrival angle (uz)  : {result.angle_mean_rad:>12.6f}")
print(f"Mean path length         : {result.dist_mean_m:>12.4f} m")
print(f"Photons beyond crit. ang : {result.reflected:>12,}")
print(f"Elapsed time             : {result.elapsed_s:>12.2f} s")

## 5. Spatial distribution at receiver plane

2D histogram of photon arrival positions — shows the beam spreading due to
forward scattering over 16 m.

In [ ]:
# rec_loc columns: [x, y, mu_x, mu_y, mu_z]
x_rec = result.rec_loc[:, 0]   # x position at receiver plane (m)
y_rec = result.rec_loc[:, 1]   # y position at receiver plane (m)

fig, ax = plt.subplots(figsize=(6, 5))
h = ax.hist2d(x_rec, y_rec, bins=80, range=[[-0.4, 0.4], [-0.4, 0.4]],
              norm=mcolors.LogNorm(), cmap="viridis")
plt.colorbar(h[3], ax=ax, label="Photon count")
ax.set_xlabel("x  (m)")
ax.set_ylabel("y  (m)")
ax.set_title(f"Arrival positions at z = {RANGE_M} m\n"
             f"({len(x_rec):,} photons reached receiver plane)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 6. Arrival angle distribution

Histogram of μ_z (cosine of arrival angle) for photons that reached the
receiver plane. Values near 1.0 are nearly on-axis (ballistic); lower
values are multiply-scattered.

In [ ]:
mu_z = result.rec_loc[:, 4]
arr_angle_deg = np.degrees(np.arccos(np.clip(mu_z, -1, 1)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# μ_z histogram
axes[0].hist(mu_z, bins=80, color="steelblue", edgecolor="none", density=True)
axes[0].set_xlabel("μ_z  (cos of arrival angle)")
axes[0].set_ylabel("Probability density")
axes[0].set_title("Arrival direction cosine distribution")
axes[0].axvline(result.angle_mean_rad, color="tomato", lw=1.5,
                label=f"mean μ_z = {result.angle_mean_rad:.4f}")
axes[0].legend()

# Arrival angle in degrees
axes[1].hist(arr_angle_deg, bins=80, color="seagreen", edgecolor="none", density=True)
axes[1].set_xlabel("Arrival angle  (deg)")
axes[1].set_ylabel("Probability density")
axes[1].set_title("Arrival angle distribution")

plt.tight_layout()
plt.show()

## 7. Path length distribution

Photons that travel longer paths have scattered more. The distribution
broadens around the geometric path length (straight-line distance = 16 m).

In [ ]:
path_lengths = result.distances_m

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(path_lengths, bins=100, color="mediumpurple", edgecolor="none", density=True)
ax.axvline(RANGE_M, color="black", lw=1.5, linestyle="--", label=f"geometric path = {RANGE_M} m")
ax.axvline(result.dist_mean_m, color="tomato", lw=1.5,
           label=f"mean path = {result.dist_mean_m:.2f} m")
ax.set_xlabel("Path length  (m)")
ax.set_ylabel("Probability density")
ax.set_title("Path length distribution at receiver plane")
ax.legend()
plt.tight_layout()
plt.show()

excess_path = result.dist_mean_m - RANGE_M
print(f"Mean excess path length: {excess_path:.4f} m  (scattering delay)")

## 8. Received power vs range (Beer-Lambert comparison)

Re-running the simulation at several receiver depths to compare the simulated
received power against the Beer-Lambert analytical prediction.

In [ ]:
ranges_m = np.array([2.0, 4.0, 8.0, 12.0, 16.0, 20.0, 24.0])
sim_power = []

for z in ranges_m:
    rx = Receiver(receiver_z_m=z, aperture_m=0.8, fov_rad=np.pi / 2)
    s = Simulation(
        medium=water, beam=beam, phase_fn=phase_fn, receiver=rx,
        n_photons=50_000, n_batches=3, backend="cpu", seed=0,
    )
    r = s.run()
    sim_power.append(r.total_power / r.n_photons)
    print(f"  z = {z:5.1f} m  →  P/P₀ = {sim_power[-1]:.5f}")

sim_power = np.array(sim_power)
bl_power = np.exp(-c * ranges_m)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(ranges_m, bl_power, "k--", label="Beer-Lambert  exp(−cz)")
ax.semilogy(ranges_m, sim_power, "o-", color="steelblue",
            label="MC simulation (including scattered photons)")
ax.set_xlabel("Range  (m)")
ax.set_ylabel("Relative received power  P/P₀")
ax.set_title("Gaussian beam in clear ocean water — power vs range")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Save results to HDF5

In [ ]:
output_path = "gaussian_beam_cpu.h5"
save_hdf5(result, output_path)
print(f"Saved to {output_path}")

# Verify round-trip
loaded = load_hdf5(output_path)
print(f"Loaded: power = {loaded['receiver']['power']:.4f}, "
      f"count = {int(loaded['receiver']['count'])}")